In [1]:
import pandas as pd

In [2]:
# 1. Define file paths
file_list = ["daily_liquidity_size_factor_panel_company_first.xlsx", "sentiment_signals.csv", "alpha_factors.csv"]

# 2. Load the dataframes into a list
dfs = []
for i, file_name in enumerate(file_list):
    if file_name.endswith('.xlsx'):
        temp_df = pd.read_excel(file_name)
    else:
        temp_df = pd.read_csv(file_name)
    
    # Ensure date is in datetime format for accurate merging
    temp_df['date'] = pd.to_datetime(temp_df['date'])
    
    # Rename 'signal' to a unique name so they don't overwrite each other
    temp_df = temp_df.rename(columns={"signal": f"signal_{i+1}"})
    
    # If there is an index column, drop it
    if temp_df.columns[0] == "": # or temp_df.columns[0].startswith('Unnamed')
        temp_df = temp_df.drop(columns=[temp_df.columns[0]])
        
    dfs.append(temp_df)

# 3. Merge files sequentially using 'inner' join
merged_df = dfs[0]
for next_df in dfs[1:]:
    merged_df = pd.merge(
        merged_df, 
        next_df, 
        on=["date", "company"], 
        how="inner"
    )

# 4. Final cleaning
merged_df = merged_df.sort_values(by=["company", "date"]).reset_index(drop=True)

# Save the result
merged_df.to_csv("merged_signals.csv", index=False)

print(f"Merged successfully! Total rows: {len(merged_df)}")
print(merged_df.head())

Merged successfully! Total rows: 66360
   year company       date  adjclose_x  Dollar Volume  Turnover    Market Cap  \
0  2015    AAPL 2015-01-02   24.214897   5.153376e+09  0.014496  3.555023e+11   
1  2015    AAPL 2015-01-05   23.532722   6.051251e+09  0.017515  3.454872e+11   
2  2015    AAPL 2015-01-06   23.534937   6.194122e+09  0.017927  3.455197e+11   
3  2015    AAPL 2015-01-07   23.864948   3.828501e+09  0.010927  3.503646e+11   
4  2015    AAPL 2015-01-08   24.781891   5.884658e+09  0.016174  3.638264e+11   

   log_dollar_volume  log_market_cap  z_log_dollar_volume  ...       high  \
0          22.065625       26.554123             2.428901  ...  27.860001   
1          22.271858       26.529995             2.277858  ...  27.162500   
2          22.294798       26.526667             2.191012  ...  26.857500   
3          21.874324       26.540292             2.305371  ...  27.049999   
4          22.230544       26.575992             2.581795  ...  28.037500   

         lo